###Introduction to Big Data Final Project

#### **Job Market Analysis Using MapReduce and Apache Spark**

This project analyzes a large job market dataset containing 500,000 rows. The goal is to apply Big Data processing concepts using both MapReduce and Apache Spark.

The project includes:
- MapReduce implementation in Python
- Apache Spark DataFrame workflow
- Data processing and exploratory data analysis
- Optimization techniques
- Optional preparation for machine learning

### **Setup and Data Loading**
In this section, we import the necessary libraries and load the dataset from Google Drive.  
The dataset contains job market information including salaries, experience, occupation, and location.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
from functools import reduce
file_path = '/content/drive/MyDrive/Big Data/job_market_dataset.csv'
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")
print("Shape of the dataset", df.shape)
df.head()

Dataset loaded successfully.
Shape of the dataset (500000, 12)


,country,city,occupation,field,years_of_experience,salary,employment_type,education_level,gender,company_size,year,month
0,Switzerland,Zurich,Operations Manager,Operations,16,359609.0,work_from_home,Master,Male,Large,2023,8
1,India,Bangalore,HR Analyst,Human Resources,12,79059.0,part_time,PhD,Male,Large,2023,5
2,Sweden,Stockholm,Software Engineer,Technology,10,258077.0,freelance,Master,Male,Enterprise,2023,9
3,South Korea,Seoul,Operations Manager,Operations,7,252282.0,part_time,Master,Female,Large,2024,12
4,United States,New York,Cloud Engineer,Technology,4,330618.0,part_time,Master,Male,Enterprise,2022,1


### **Data Understanding**
In this section, we explore the structure of the dataset.  
We examine the schema, preview sample data, and determine the number of rows and columns.  
This step is important to understand the dataset before applying Big Data processing techniques.

In [4]:
#display column names
print("Columns:")
print(df.columns)

#display basic information
print("\nDataset Info:")
df.info()

#display number of rows and columns
print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Columns:
Index(['country', 'city', 'occupation', 'field', 'years_of_experience',
       'salary', 'employment_type', 'education_level', 'gender',
       'company_size', 'year', 'month'],
      dtype='object')

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   country              500000 non-null  object 
 1   city                 500000 non-null  object 
 2   occupation           500000 non-null  object 
 3   field                500000 non-null  object 
 4   years_of_experience  500000 non-null  int64  
 5   salary               500000 non-null  float64
 6   employment_type      500000 non-null  object 
 7   education_level      500000 non-null  object 
 8   gender               500000 non-null  object 
 9   company_size         500000 non-null  object 
 10  year                 500000 non-null  int64  
 1

###Interpretation
The dataset contains 500,000 rows and 12 columns, making it suitable for Big Data processing. It includes both numerical features (such as salary and years of experience) and categorical features (such as occupation, country and employment type). There are no missing values, which simplifies the data processing step.

###Part 1: **MapReduce Implementation**
The goal of this section is to apply the MapReduce concept using Python.

For this dataset, the MapReduce task is to calculate the average salary for each occupation.


*   The mapper converts each row into a key-value pair: (occupation, salary).
* The shuffle step groups salaries by occupation.
* The reducer calculates the average salary for each occupation.
* Chunk-based processing is simulated to reflect how distributed systems process large datasets.



In [5]:
#select only the columns needed for the MapReduce task
mapreduce_data = df[['occupation', 'salary']]
#preview the selected data
mapreduce_data.head()

,occupation,salary
0,Operations Manager,359609.0
1,HR Analyst,79059.0
2,Software Engineer,258077.0
3,Operations Manager,252282.0
4,Cloud Engineer,330618.0


In [6]:
#Mapper function
#Input: one row from the dataset
#Output: a key-value pair in the form (occupation, salary)
def mapper(row):
  occupation = row['occupation']
  salary = row['salary']
  return (occupation, salary)

In [7]:
#Apply the mapper function using map()
#Each row becomes a key-value pair
mapped_data = list(map(mapper, mapreduce_data.to_dict('records')))
#display first 10 mapped outputs
mapped_data[:10]

[('Operations Manager', 359609.0),
 ('HR Analyst', 79059.0),
 ('Software Engineer', 258077.0),
 ('Operations Manager', 252282.0),
 ('Cloud Engineer', 330618.0),
 ('HR Analyst', 104059.0),
 ('UX Designer', 166377.0),
 ('Marketing Specialist', 188237.0),
 ('Business Analyst', 211953.0),
 ('Business Analyst', 86874.0)]

###Mapper Interpretation
The mapper transforms each row of the dataset into a key-value pair, where the key is the occupation and the value is the salary. This step prepares the data for grouping by occupation in the next phase of the MapReduce process.

In [8]:
from collections import defaultdict
#Shuffle phase: group all salaries by occupation
grouped_data = defaultdict(list)
for occupation, salary in mapped_data:
  grouped_data[occupation].append(salary)
#display first 5 grouped results
list(grouped_data.items())[:5]

[('Operations Manager',
  [359609.0,
   252282.0,
   103021.0,
   226393.0,
   151562.0,
   67783.0,
   206682.0,
   180278.0,
   109221.0,
   208117.0,
   310837.0,
   240435.0,
   185121.0,
   118845.0,
   212428.0,
   251152.0,
   221145.0,
   82869.0,
   195521.0,
   159444.0,
   253519.0,
   255144.0,
   83890.0,
   185432.0,
   198891.0,
   260290.0,
   169672.0,
   215168.0,
   176846.0,
   79630.0,
   243608.0,
   147365.0,
   267518.0,
   191005.0,
   219009.0,
   259281.0,
   284974.0,
   123303.0,
   63763.0,
   108467.0,
   266392.0,
   189795.0,
   157028.0,
   150815.0,
   73150.0,
   169328.0,
   269960.0,
   223344.0,
   206202.0,
   318725.0,
   147549.0,
   180822.0,
   140852.0,
   160396.0,
   136410.0,
   197029.0,
   269471.0,
   298251.0,
   353639.0,
   370000.0,
   370000.0,
   236204.0,
   335586.0,
   206521.0,
   370000.0,
   343281.0,
   370000.0,
   345661.0,
   238825.0,
   329338.0,
   203459.0,
   268230.0,
   225179.0,
   145465.0,
   70865.0,
   25112

###Shuffle Phase Interpretation
The shuffle phase groups all salary values by occupation. This means that every occupation now has a list of salaries connected to it. This step is necessary before applying the reducer, because the reducer needs all values for the same key in order to calculate the average salary.

In [9]:
#Reducer function
#Input: one occupation and its list of salaries
#Output: occupation and average salary
def reducer(item):
  occupation, salaries = item
  average_salary = sum(salaries) / len(salaries)
  return (occupation, average_salary)

In [10]:
#Apply reducer to grouped data
reduced_data = list(map(reducer, grouped_data.items()))
#display first 10 reduced results
reduced_data[:10]

[('Operations Manager', 213451.235518732),
 ('HR Analyst', 144604.00033688667),
 ('Software Engineer', 223677.86047125445),
 ('Cloud Engineer', 242826.7555763195),
 ('UX Designer', 192173.97958988577),
 ('Marketing Specialist', 156345.1849212845),
 ('Business Analyst', 191619.7314657886),
 ('Financial Analyst', 202585.41912186722),
 ('Data Scientist', 233261.97111748127),
 ('Data Analyst', 179691.18065597318)]

### Reducer Interpretation
The reducer calculates the average salary for each occupation by summing all salaries for that occupation and dividing by the number of records. The output shows each occupation with its average salary, which helps identify salary differences between job roles.

In [11]:
#sort occupations by average salary in descending order
sorted_mapreduce_results = sorted(reduced_data, key=lambda x: x[1], reverse=True)
#display top 10 highest-paying occupations
sorted_mapreduce_results[:10]

[('AI Engineer', 252158.15810768752),
 ('Product Manager', 251276.937323708),
 ('Cloud Engineer', 242826.7555763195),
 ('Data Scientist', 233261.97111748127),
 ('Software Engineer', 223677.86047125445),
 ('Operations Manager', 213451.235518732),
 ('Financial Analyst', 202585.41912186722),
 ('UX Designer', 192173.97958988577),
 ('Business Analyst', 191619.7314657886),
 ('Data Analyst', 179691.18065597318)]

### MapReduce Results Interpretation
The results show the top-paying occupations based on average salary.

Roles such as AI Engineer, Product Manager and Cloud Engineer have the highest average salaries, indicating that technical and specialized roles tend to be more highly compensated.

This analysis highlights how salary levels vary significantly depending on the occupation, with data-related and enginnering roles among the highest paying.

### MapReduce Explanation
The mapper function processes each row of the dataset and produces key-value pairs in the form (occupation, salary).
During the shuffle phase, all values associated with the same key (occupation) are grouped together.
The reducer then aggregates these values by calculating the average salary for each occupation.
This process demonstrates how large datasets can be processed efficiently by dividing tasks into smaller steps and combining results.

### **Chunk-Based Processing with 'functools.reduce()'**
To better simulate distributed Big Data processing, the dataset is divided into smaller chunks. Each chunk is processed seperately using the mapper logic, similar to how different machines process parts of a dataset in a distributed system.

The 'functools.reduce()' function is then used to combine the partial grouped results from all chunks into one final grouped dataset.

In [12]:
chunk_size = 50000
chunks = [
    mapreduce_data[i:i + chunk_size]
    for i in range(0, len(mapreduce_data), chunk_size)
]
print("Number of chunks:", len(chunks))

Number of chunks: 10


In [13]:
#Process each chunk seperately
def process_chunk(chunk):
  mapped = list(map(mapper, chunk.to_dict('records')))
  #shuffle phase per chunk
  local_group = {}
  for occupation, salary in mapped:
    if occupation not in local_group:
      local_group[occupation] = []
    local_group[occupation].append(salary)
  return local_group
#apply to all chunks
chunk_results = list(map(process_chunk, chunks))
len(chunk_results)

10

### Chunk Processing Interpretation
Each chunk is processed independently using the mapper and local grouping logic. This simulates how distributed systems divide large datasets into smaller parts and process them in parallel across multiple machines.

In [14]:
def merge_groups(group1, group2):
  for occupation, salaries in group2.items():
    if occupation not in group1:
      group1[occupation] = []
    group1[occupation].extend(salaries)
  return group1
#combine all chunk results
final_grouped_data = reduce(merge_groups, chunk_results)
list(final_grouped_data.items())[:5]

[('Operations Manager',
  [359609.0,
   252282.0,
   103021.0,
   226393.0,
   151562.0,
   67783.0,
   206682.0,
   180278.0,
   109221.0,
   208117.0,
   310837.0,
   240435.0,
   185121.0,
   118845.0,
   212428.0,
   251152.0,
   221145.0,
   82869.0,
   195521.0,
   159444.0,
   253519.0,
   255144.0,
   83890.0,
   185432.0,
   198891.0,
   260290.0,
   169672.0,
   215168.0,
   176846.0,
   79630.0,
   243608.0,
   147365.0,
   267518.0,
   191005.0,
   219009.0,
   259281.0,
   284974.0,
   123303.0,
   63763.0,
   108467.0,
   266392.0,
   189795.0,
   157028.0,
   150815.0,
   73150.0,
   169328.0,
   269960.0,
   223344.0,
   206202.0,
   318725.0,
   147549.0,
   180822.0,
   140852.0,
   160396.0,
   136410.0,
   197029.0,
   269471.0,
   298251.0,
   353639.0,
   370000.0,
   370000.0,
   236204.0,
   335586.0,
   206521.0,
   370000.0,
   343281.0,
   370000.0,
   345661.0,
   238825.0,
   329338.0,
   203459.0,
   268230.0,
   225179.0,
   145465.0,
   70865.0,
   25112

### Reduce Function Interpretation
The 'functools.reduce()' function combines the partial results from all chunks into one final grouped dataset. This simulates the final aggregation stage in a distributed system, where results from different workers are merged before producing the final output.

In [15]:
#Calculate final average salary per occupation after combining all chunks
chunk_average_salary = {
    occupation: sum(salaries) / len(salaries)
    for occupation, salaries in final_grouped_data.items()
}
#sort results by average salary in descending order
chunk_sorted_results = sorted(chunk_average_salary.items(), key=lambda x: x[1], reverse=True)
chunk_sorted_results[:10]

[('AI Engineer', 252158.15810768752),
 ('Product Manager', 251276.937323708),
 ('Cloud Engineer', 242826.7555763195),
 ('Data Scientist', 233261.97111748127),
 ('Software Engineer', 223677.86047125445),
 ('Operations Manager', 213451.235518732),
 ('Financial Analyst', 202585.41912186722),
 ('UX Designer', 192173.97958988577),
 ('Business Analyst', 191619.7314657886),
 ('Data Analyst', 179691.18065597318)]

### Chunk-Based Mapreduce Results Interpretation
After combining all chunks using 'functools.reduce()', the final average salary was calculated for each occupation. The results confirm that the chunk-based MapReduce simulation produces the same type of output as the original MapReduce process, while better representing how distributed systems process large datasets.

### Parallelization Explanation
In a real Big Data environment, the dataset would be divided into smaller chunks and processed across multiple machines. Each machine would apply the mapper function to its assigned chunk and create local grouped results. Then, the system would shuffle and merge results with the same key. Finally, reducers would calculate the final average salary for each occupation.

As the dataset becomes larger, parallel processing improves performance because multiple chunks can be processed at the same time instead of processing the entire dataset on one machine.

## Part 2: **Apache Spark Workflow**
In this section, Apache Spark is used to process and analyze the dataset efficiently. Spark allows large-scale data processing using distributed computing and DataFrames.
We will perform data filtering, transformation, aggregation and analysis using PySpark.

In [16]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Job Market Analysis").getOrCreate()
print("Spark session created successfully.")

Spark session created successfully.


In [17]:
#load the csv file into a Spark DataFrame
spark_df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)
spark_df.show(5)

+-------------+---------+------------------+---------------+-------------------+--------+---------------+---------------+------+------------+----+-----+
|      country|     city|        occupation|          field|years_of_experience|  salary|employment_type|education_level|gender|company_size|year|month|
+-------------+---------+------------------+---------------+-------------------+--------+---------------+---------------+------+------------+----+-----+
|  Switzerland|   Zurich|Operations Manager|     Operations|                 16|359609.0| work_from_home|         Master|  Male|       Large|2023|    8|
|        India|Bangalore|        HR Analyst|Human Resources|                 12| 79059.0|      part_time|            PhD|  Male|       Large|2023|    5|
|       Sweden|Stockholm| Software Engineer|     Technology|                 10|258077.0|      freelance|         Master|  Male|  Enterprise|2023|    9|
|  South Korea|    Seoul|Operations Manager|     Operations|                  7|25

### Spark Data Loading Interpretation
The dataset was loaded into a Spark DataFrame. Using Spark DataFrames allows the data to be processed efficiently in a distributed environment, which is important for large datasets.

In [18]:
#display the schema of the Spark DataFrame
spark_df.printSchema()
#count rows and columns
row_count = spark_df.count()
column_count = len(spark_df.columns)
print("Number of rows:", row_count)
print("Number of columns:", column_count)

root
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- field: string (nullable = true)
 |-- years_of_experience: integer (nullable = true)
 |-- salary: double (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- company_size: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Number of rows: 500000
Number of columns: 12


### Spark Data Understanding Interpretation
The schema shows that the dataset contains a mix of categorical and numerical features. Categorical variables include columns such as country, city, occupation, field, employment type, education level, gender and company size. Numerical variables include years of experience, salary, year and month.

All columns are marked as nullable, meaning they can contain missing values, although earlier inspection showed no missing data. The dataset contains 500,000 rows and 12 columns, confirming that is large enough to require efficient processing using Spark.

### **Data Filtering**
In this step, we filter the dataset to focus on high-paying jobs. This helps us analyze patterns among higher salary roles and identify which occupations and conditions are associated with better compensation.

In [19]:
#filter jobs with salary greater than 200,000
high_salary_df = spark_df.filter(spark_df.salary > 200000)
high_salary_df.show(5)
print("Number of high-paying jobs:", high_salary_df.count())

+--------------------+---------+------------------+----------+-------------------+--------+---------------+---------------+------+------------+----+-----+
|             country|     city|        occupation|     field|years_of_experience|  salary|employment_type|education_level|gender|company_size|year|month|
+--------------------+---------+------------------+----------+-------------------+--------+---------------+---------------+------+------------+----+-----+
|         Switzerland|   Zurich|Operations Manager|Operations|                 16|359609.0| work_from_home|         Master|  Male|       Large|2023|    8|
|              Sweden|Stockholm| Software Engineer|Technology|                 10|258077.0|      freelance|         Master|  Male|  Enterprise|2023|    9|
|         South Korea|    Seoul|Operations Manager|Operations|                  7|252282.0|      part_time|         Master|Female|       Large|2024|   12|
|       United States| New York|    Cloud Engineer|Technology|        

### Filtering Interpretation
The filtered dataset shows that a significant portion of jobs (250,511 out of 500,000) have salaries above 200,000, indicating that high-paying roles are relatively common in this dataset. From the sample output, high-paying jobs are observed across different countries such as Switzerland, Sweden, the United States and the United Arab Emirates, suggesting that high salaries are not limited to a signle region but are globally distributed.

These roles are often associated with mid to high levels of experience (4 and 16 years years), indicating that experience plays a key role in achieving higher salaries.

Additionally, many of these positions are in fields like "Technology" and "Operations", reinforcing the idea that technical and managerial roles tend to offer higher compensation.

### **Column Selection**
In this step, we select only the most relevant columns for analysis. Focusing on key features such as occupation, salary, years of experience, level of education and country helps simplify the dataset and improves processing efficiency while keeping the most important information for analysis.

In [20]:
#select important columns for analysis
selected_df = spark_df.select(
    "occupation",
    "salary",
    "years_of_experience",
    "country",
    "education_level"
)
selected_df.show(5)

+------------------+--------+-------------------+-------------+---------------+
|        occupation|  salary|years_of_experience|      country|education_level|
+------------------+--------+-------------------+-------------+---------------+
|Operations Manager|359609.0|                 16|  Switzerland|         Master|
|        HR Analyst| 79059.0|                 12|        India|            PhD|
| Software Engineer|258077.0|                 10|       Sweden|         Master|
|Operations Manager|252282.0|                  7|  South Korea|         Master|
|    Cloud Engineer|330618.0|                  4|United States|         Master|
+------------------+--------+-------------------+-------------+---------------+
only showing top 5 rows


### Column Selection Interpretation
The dataset has been refined to include the most relevant features for salary analysis.
* Occupation helps identify high-paying roles.
* Salary is the main variable of interest.
* Years of experience allows analysis of how experience influences income.
* Country provides insight into geographical salary differences.
* Education level adds another important dimension, enabling analysis of how academic qualifications impact salary.

### Feature Engineering: Salary Category
In this step, a new feature called 'salary_category' is created. This categorizes saalries into three groups: Low, Medium and High. This transormation simplifies analysis and helps identify patterns more easily across different salary levels.

In [21]:
from pyspark.sql.functions import when
enhanced_df = selected_df.withColumn(
    "salary_category",
    when(selected_df.salary < 100000, "Low")
    .when((selected_df.salary >= 100000) & (selected_df.salary < 200000), "Medium")
    .otherwise("High")
)
enhanced_df.show(5)

+------------------+--------+-------------------+-------------+---------------+---------------+
|        occupation|  salary|years_of_experience|      country|education_level|salary_category|
+------------------+--------+-------------------+-------------+---------------+---------------+
|Operations Manager|359609.0|                 16|  Switzerland|         Master|           High|
|        HR Analyst| 79059.0|                 12|        India|            PhD|            Low|
| Software Engineer|258077.0|                 10|       Sweden|         Master|           High|
|Operations Manager|252282.0|                  7|  South Korea|         Master|           High|
|    Cloud Engineer|330618.0|                  4|United States|         Master|           High|
+------------------+--------+-------------------+-------------+---------------+---------------+
only showing top 5 rows


In [22]:
#Count how many jobs per salary category
enhanced_df.groupBy("salary_category").count().show()

+---------------+------+
|salary_category| count|
+---------------+------+
|           High|250514|
|            Low| 51259|
|         Medium|198227|
+---------------+------+



### Salary Category Interpretation
The distribution of jobs across salary categories shows that high-paying jobs dominate the dataset with 250,514 positions classified as "High". Medium-paying jobs follow with 198,227 entries while low-paying jobs are significantly fewer at 51,259.
This indicates that the dataset is skewed toward higher salary ranges, suggesting that many roles in this dataset offer competitive compensation.

Overall, this distribution highlights that a large portion of the job market in this dataset is concentrated in medium to high salary ranges, reinforcing the importance of experience, skills and specialization in achieving higher pay.

### Feature Engineering: Experience Level
In this step, a new feature called 'experience_level' is created. This categorizes professionals into Junior, Mid and Senior levels based on their years of experience. This helps simplify analysis and allows us to better understand how experience impacts salary.

In [23]:
from pyspark.sql.functions import when
enhanced_df = enhanced_df.withColumn(
    "experience_level",
    when(enhanced_df.years_of_experience < 3, "Junior")
    .when((enhanced_df.years_of_experience >= 3) & (enhanced_df.years_of_experience < 10), "Mid")
    .otherwise("Senior")
)
enhanced_df.show(5)

+------------------+--------+-------------------+-------------+---------------+---------------+----------------+
|        occupation|  salary|years_of_experience|      country|education_level|salary_category|experience_level|
+------------------+--------+-------------------+-------------+---------------+---------------+----------------+
|Operations Manager|359609.0|                 16|  Switzerland|         Master|           High|          Senior|
|        HR Analyst| 79059.0|                 12|        India|            PhD|            Low|          Senior|
| Software Engineer|258077.0|                 10|       Sweden|         Master|           High|          Senior|
|Operations Manager|252282.0|                  7|  South Korea|         Master|           High|             Mid|
|    Cloud Engineer|330618.0|                  4|United States|         Master|           High|             Mid|
+------------------+--------+-------------------+-------------+---------------+---------------+-

In [24]:
#Count how many jobs per experience level
enhanced_df.groupBy("experience_level").count().show()

+----------------+------+
|experience_level| count|
+----------------+------+
|          Senior|297234|
|             Mid|141849|
|          Junior| 60917|
+----------------+------+



### Experience Level Interpretation
The distribution of jobs across experience levels shows a clear imbalnce. Senior-level positions dominate the dataset with 297,234 jobs, followed by Mid-level roles with 141,849 jobs while Junior positions are significantly fewer at 60,917.

This suggests that the job market represented in this dataset is heavily skewed toward experienced professionals indicating that higher experience is more in demand. Additionally, since senior roles are the majority, it is liekly that a large portion of high-apying jobs are associated with higher experience levels.

This insight highlights the strong relationship between experience and job availability and suggests that career progression plays a key role in accessing more opportunities and higher salaries.

### **Advanced Operations**

### Basic Aggregation: Average Salary by Occupation
This analysis calculates the average salary for each occupation to identify the highest-paying job roles.

In [25]:
from pyspark.sql.functions import avg
avg_salary_job = enhanced_df.groupBy("occupation").agg(avg("salary").alias("avg_salary")).orderBy("avg_salary", ascending=False)
avg_salary_job.show(10)

+------------------+------------------+
|        occupation|        avg_salary|
+------------------+------------------+
|       AI Engineer|252158.15810768752|
|   Product Manager|  251276.937323708|
|    Cloud Engineer| 242826.7555763195|
|    Data Scientist|233261.97111748127|
| Software Engineer|223677.86047125445|
|Operations Manager|  213451.235518732|
| Financial Analyst|202585.41912186722|
|       UX Designer|192173.97958988577|
|  Business Analyst| 191619.7314657886|
|      Data Analyst|179691.18065597318|
+------------------+------------------+
only showing top 10 rows


### Occupation-Based Salary Interpretation
The result show that the highest average salaries are concentrated in technical and leadership roles. AI Engineer and Product Manager are the top-paying occupations, both exceeding 250,000 in average salary. Other high-paying roles include Cloud Engineer, Data Scientist and Software Engineer which are all part of the technology sector. This indicates that jobs related to advanced technologies, data and cloud computing are among the most financially rewarding. Additionally, management roles such as Product Manager and Operations Manager also rank highly, suggesting that leadership and strategic responsibilities are strongly linked to higher compensation.

Overall, this analysis demonstrates that both technical expertise and managerial responsibility play a key role in achieving higher salaries in the job market.

While average salary provides a general overview, it does not capture the full distribution of salaries.  
Therefore, we extend the analysis by including minimum, maximum, and job count to gain deeper insights.

### Advanced Aggregation: Salary Distribution by Occupation
In this step, we compute multiple statistics for each occupation including,
* Average salary
* Minimum salary
* Maximum salary
* Number of jobs
This provides a more comprehensive understanding of salary distribution across different roles.

In [26]:
from pyspark.sql.functions import avg, min, max, count
salary_stats_job = enhanced_df.groupBy("occupation").agg(
    avg("salary").alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    count("*").alias("job_count")).orderBy("avg_salary", ascending=False)
salary_stats_job.show(10)

+------------------+------------------+----------+----------+---------+
|        occupation|        avg_salary|min_salary|max_salary|job_count|
+------------------+------------------+----------+----------+---------+
|       AI Engineer|252158.15810768752|   12000.0|  370000.0|    41769|
|   Product Manager|  251276.937323708|   12000.0|  370000.0|    41834|
|    Cloud Engineer| 242826.7555763195|   12237.0|  370000.0|    41739|
|    Data Scientist|233261.97111748127|   12000.0|  370000.0|    41513|
| Software Engineer|223677.86047125445|   12247.0|  370000.0|    41676|
|Operations Manager|  213451.235518732|   12000.0|  370000.0|    41640|
| Financial Analyst|202585.41912186722|   12000.0|  370000.0|    41816|
|       UX Designer|192173.97958988577|   12000.0|  370000.0|    41842|
|  Business Analyst| 191619.7314657886|   12215.0|  370000.0|    41302|
|      Data Analyst|179691.18065597318|   12000.0|  370000.0|    41770|
+------------------+------------------+----------+----------+---

### Advanced Aggregation Interpretation
The results provide a comprehensive view of salary distribution across different occupations.
AI Engineer and Product Manager remain the highest-paying roles based on average salary, confirming that both advanced technical expertise and managerial responsibility are highly valued in the job market.
Although all occupations share a similar maximum salary (370,000), the differences in average salary indicate that higher-paying roles are more consistently compensated at elevated levels.  
The wide gap between minimum and maximum salaries across all occupations highlights significant salary variability. This suggests that factors such as experience, company size, and geographic location strongly influence earnings within the same role.
Additionally, the job_count values are relatively similar across occupations, indicating that these roles are comparably represented in the dataset. This allows for fair comparison between them without bias from uneven sample sizes.

Overall, this analysis shows that while many roles can reach high salary levels, occupations in technology and management are more likely to consistently offer higher compensation.

### Salary by Education and Experience Level
This analysis examines how salary varies based on both education level and experience level. By combining these two factors, we can better understand how academic qualifications and professional experience interact to influence salary.

In [27]:
from pyspark.sql.functions import avg
salary_edu_exp = enhanced_df.groupBy("education_level", "experience_level").agg(avg("salary").alias("avg_salary")).orderBy("education_level", "experience_level")
salary_edu_exp.show()

+---------------+----------------+------------------+
|education_level|experience_level|        avg_salary|
+---------------+----------------+------------------+
|       Bachelor|          Junior|116692.32593982443|
|       Bachelor|             Mid| 181513.1514817951|
|       Bachelor|          Senior|217343.16443982112|
|    High School|          Junior|100459.41348212548|
|    High School|             Mid| 154986.9853003176|
|    High School|          Senior| 186949.5215383787|
|         Master|          Junior| 134466.0605981625|
|         Master|             Mid|207510.01371976882|
|         Master|          Senior|245848.50996327898|
|            PhD|          Junior| 149557.3763292635|
|            PhD|             Mid|231356.02446161714|
|            PhD|          Senior| 269948.5978090976|
+---------------+----------------+------------------+



In [28]:
salary_edu_exp.orderBy("avg_salary", ascending=False).show()

+---------------+----------------+------------------+
|education_level|experience_level|        avg_salary|
+---------------+----------------+------------------+
|            PhD|          Senior| 269948.5978090976|
|         Master|          Senior|245848.50996327898|
|            PhD|             Mid|231356.02446161714|
|       Bachelor|          Senior|217343.16443982112|
|         Master|             Mid|207510.01371976882|
|    High School|          Senior| 186949.5215383787|
|       Bachelor|             Mid| 181513.1514817951|
|    High School|             Mid| 154986.9853003176|
|            PhD|          Junior| 149557.3763292635|
|         Master|          Junior| 134466.0605981625|
|       Bachelor|          Junior|116692.32593982443|
|    High School|          Junior|100459.41348212548|
+---------------+----------------+------------------+



### Education and Experience-Based Salary Interpretation
The results have a clear relationship between both education level and experience level in determining salary. Across all education levels, salaries consistently increase from Junior to Mid to Senior positions, confirming that experience is a strong and reliable driver of income. At the same time, education level also plays a significant role. For each experience level, individuals with higher degrees tend to earn more. The highest overall salaries are observed among Senior professionals with advanced degrees, particularly PhDs and Master's degrees, indicating that the combination of high experience and advanced education leads to the greatest earning potential.
However, at lower experience levels, the differences between education levels are less pronounced, suggesting that early in a career, experience may matter more than education.

Overall, this analysis highlights that while both education and experience influence salary, experience has a more consistent impact while education provides an additional advantage especially at higher career levels.

### **Exploratory Data Analysis (EDA)**
In this section, we explore general patterns in the dataset to better understand the job market distribution.

In [29]:
from pyspark.sql.functions import count
top_countries = enhanced_df.groupBy("country").agg(count("*").alias("job_count")).orderBy("job_count", ascending=False)
top_countries.show(10)

+--------------+---------+
|       country|job_count|
+--------------+---------+
| United States|    45238|
|        Brazil|    23010|
|         Italy|    22971|
|        Mexico|    22964|
|       Ireland|    22959|
|United Kingdom|    22871|
|         Spain|    22839|
|       Germany|    22838|
|        Sweden|    22793|
|        Canada|    22769|
+--------------+---------+
only showing top 10 rows


### Country-Based Job Distribution Interpretation
The results show that the United States has the highest number of job entries in the dataset, significantly exceeding other countries.
Other countries such as Brazil, Italy, Mexico, and Ireland also have high job counts, but at a noticeably lower level compared to the United States.
This suggests that the dataset is somewhat dominated by job listings from the United States, which may indicate either a stronger job market presence or a data collection bias toward that region.
Additionally, the relatively similar job counts among several European and Latin American countries suggest a more balanced distribution of opportunities outside the top country.

Overall, this analysis highlights geographical differences in job availability and suggests that regional factors may influence job distribution in the dataset.

In [30]:
enhanced_df.describe().show()

+-------+-----------+-----------------+-------------------+-------------+---------------+---------------+----------------+
|summary| occupation|           salary|years_of_experience|      country|education_level|salary_category|experience_level|
+-------+-----------+-----------------+-------------------+-------------+---------------+---------------+----------------+
|  count|     500000|           500000|             500000|       500000|         500000|         500000|          500000|
|   mean|       NULL|    207019.212078|          11.865168|         NULL|           NULL|           NULL|            NULL|
| stddev|       NULL|85867.30737286204|  7.178829110861335|         NULL|           NULL|           NULL|            NULL|
|    min|AI Engineer|          12000.0|                  0|    Australia|       Bachelor|           High|          Junior|
|    max|UX Designer|         370000.0|                 25|United States|            PhD|         Medium|          Senior|
+-------+-------

### Summary Statistics Interpretation
The summary statistics provide an overview of the dataset, including mean, minimum, and maximum values.
The average salary is relatively high, confirming that the dataset is skewed toward well-compensated roles.  
The wide range between minimum and maximum salaries indicates significant variability in income across different jobs and experience levels.

### **Optimization:** Column Reduction
In Big Data processing, it is important to reduce the size of the dataset by removing unnecessary columns.
This improves performance and reduces memory usage, especially when working with large datasets.

In [31]:
optimized_df = enhanced_df.select(
    "occupation",
    "salary",
    "experience_level",
    "education_level",
    "salary_category"
)
optimized_df.show(5)

+------------------+--------+----------------+---------------+---------------+
|        occupation|  salary|experience_level|education_level|salary_category|
+------------------+--------+----------------+---------------+---------------+
|Operations Manager|359609.0|          Senior|         Master|           High|
|        HR Analyst| 79059.0|          Senior|            PhD|            Low|
| Software Engineer|258077.0|          Senior|         Master|           High|
|Operations Manager|252282.0|             Mid|         Master|           High|
|    Cloud Engineer|330618.0|             Mid|         Master|           High|
+------------------+--------+----------------+---------------+---------------+
only showing top 5 rows


### Optimization Interpretation
Reducing the number of columns helps improve performance when processing large datasets.
By keeping only the relevant features, we reduce memory usage and make computations faster.  
This is especially important in Big Data environments where datasets can contain millions of rows and many unnecessary attributes.
Column reduction ensures that only useful data is processed, which improves efficiency and scalability of the analysis.

### **Machine Learning Preparation**
The dataset is split into training and testing sets to prepare it for potential machine learning applications.

In [32]:
#split dataset into training (80%) and testing (20%)
train_df, test_df = enhanced_df.randomSplit([0.8, 0.2], seed=42)
print("Training set size:", train_df.count())
print("Testing set size:", test_df.count())

Training set size: 399788
Testing set size: 100212


### Train/Test Split Interpretation
The dataset was successfully divided into training and testing sets, with approximately 80% of the data (399,788 records) used for training and 20% (100,212 records) used for testing.
This balanced split ensures that the training set is large enough to learn meaningful patterns, while the testing set remains sufficiently representative for evaluating model performance.

This preparation is important in Big Data contexts, where large datasets are commonly used to build and validate scalable machine learning models.

## Part 3: **Final Analysis and Discussion**

### Key Insights
The analysis revealed several important patterns in the job market dataset.
First, salaries are strongly influenced by occupation. Technical roles such as AI Engineer, Cloud Engineer, and Data Scientist consistently offer higher average salaries compared to other roles.
Second, experience plays a critical role in salary growth. Salaries increase significantly from Junior to Senior levels, showing that career progression is a major factor in earning potential.
Third, education also impacts salary, particularly at higher experience levels. Individuals with advanced degrees such as Master's and PhD tend to earn more, especially in senior positions.
Finally, the dataset shows a concentration of high-paying jobs, indicating that many roles represented require specialized skills or experience.

---

### Limitations of the Analysis
Despite these insights, there are several limitations.
The dataset may contain bias, as certain countries (such as the United States) are overrepresented. This may affect the generalizability of the results.
Additionally, the analysis does not include other important factors that influence salary, such as specific skills, company reputation, or industry demand.
Another limitation is that salary ranges were simplified into categories, which may reduce the precision of the analysis.

---

### Scalability in a Big Data Context
The solution is designed to scale effectively in a real-world Big Data environment.
The use of MapReduce demonstrates how large datasets can be processed in parallel by dividing data into chunks and aggregating results efficiently.
Apache Spark further improves scalability by enabling distributed data processing, in-memory computation, and optimized query execution.
Techniques such as column reduction improve performance and reduce memory usage, which is essential when working with large-scale datasets.
Overall, the approach used in this project can handle much larger datasets and can be extended to real-world Big Data applications.